# Workshop: Scientific Programming with Python using Jupyter Notebooks

<div>
<img src="https://www.python.org/static/img/python-logo.png" width="350"/>
<img src="https://www.gosmarter.ai/blog/glossary/what-is-jupyter/jupyter_hu_b6cacecfb7fece90.webp" width="250"/>
<img src="https://www.kim.uni-konstanz.de/fileadmin/user_upload/csm_bwHPC_logo_quer_transparent_bb75ad50f1.png" width="350"/>
</div> 

# Agenda

- **Introduction** <br>
    - Format of this Workshop
    - Project Jupyter 
    - Accessing the JupyterLab Server <br><br>
- **Part 1: Data Management, Data Visualization & Data Science Basics** <br>
    - Clustering
    - Dimensionality Reduction <br><br>
- **Part 2: Data Science and Machine Learning** <br>
    - *Part 2.0* Neural Network Basics
        - Artificial Neural Networks
        - Activation functions
    - *Part 2.1:* NNs with Pytorch
        - Regression task with NNs
        - Classification with NNs
        - Convolutional NNs
    - *Part 2.2:* NNs with Tensorflow
        - Denoising with NNs<br><br>
- **Part 3: Framework comparison exercise** <br>
    - Data Preparation
    - Tensorboard
    - NN with Pytorch
    - NN with Tensorflow
<br><br>
- **References**


# Part 3: Framework comparison exercise

## Exercise: Using PyTorch and Tensorflow to create th same NN

Now that you have had different examples for NNs with both PyTorch and Tensorflow you have the opportunity to create a NN with both frameworks. For this, we want to create a classifier NN for our penguin dataset. We recommend to use the notebooks ML_tensorflow.ipynb and ML_pytorch.ipynb as reference when creating the NNs. We also provide the notebook example_solution.ipynb, which provides an exemplary solution to this notebook's exercises. That notebook is intended for a later comparison with your own results, as possible future reference point and as a guide in case you get *sincerely* stuck. However we strongly recommend you try to find a solution with the introductory notebooks and asking for guidance by the course tutor(s)!

### Palmer-Penguins Dataset

So let's begin with a close look at the dataset again:

In [ ]:
from copy import deepcopy
import matplotlib as mpl
from matplotlib import pyplot as plt
import math
import numpy as np
import torch

In [ ]:
from palmerpenguins import load_penguins

penguins = load_penguins()

#### Data exploration

In [ ]:
# Dropping all rows that contain not assigned values (N/A) for any of
# the properties
penguins_dropanyna = penguins.dropna(how="any", ignore_index=True)

In [ ]:
penguins_dropanyna

To get a nice overview of the data, we're going to create plots for multiple property pairings.

In [ ]:
key_list = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
key_pairs = list()

while len(key_list) > 0:
    tmp_key = key_list.pop()
    for elem in key_list:
        key_pairs.append([tmp_key, elem])

In [ ]:
key_pairs

The plots will first be colored by our classification target: species.

In [ ]:
groups = penguins_dropanyna.groupby('species')

fig, ax = plt.subplots(2,3)
fig.set_size_inches(15,10)
init = True

for name, group in groups:    
    for n, pair in enumerate(key_pairs):
        ax[(n//3), (n%3)].scatter(x = group[pair[0]], y = group[pair[1]], label = name)
        
        if init == True:
            ax[(n//3), (n%3)].set_xlabel(pair[0])
            ax[(n//3), (n%3)].set_ylabel(pair[1])

    init = False

n = ax.shape[0]
m = ax.shape[1]

for a in range(n):
    for b in range(m):
        ax[a,b].legend()

plt.legend()
plt.show()

Another property we could train a NN to determine could also be the penguins' sex. So let's plot that respective data as well:

In [ ]:
groups = penguins_dropanyna.groupby('sex')

fig, ax = plt.subplots(2,3)
fig.set_size_inches(15,10)
init = True

for name, group in groups:    
    for n, pair in enumerate(key_pairs):
        ax[(n//3), (n%3)].scatter(x = group[pair[0]], y = group[pair[1]], label = name)
        
        if init == True:
            ax[(n//3), (n%3)].set_xlabel(pair[0])
            ax[(n//3), (n%3)].set_ylabel(pair[1])

    init = False

n = ax.shape[0]
m = ax.shape[1]

for a in range(n):
    for b in range(m):
        ax[a,b].legend()
    
plt.show()

In [ ]:
groups = penguins_dropanyna.groupby('island')

fig, ax = plt.subplots(2,3)
fig.set_size_inches(15,10)
init = True

for name, group in groups:    
    for n, pair in enumerate(key_pairs):
        ax[(n//3), (n%3)].scatter(x = group[pair[0]], y = group[pair[1]], label = name)
        
        if init == True:
            ax[(n//3), (n%3)].set_xlabel(pair[0])
            ax[(n//3), (n%3)].set_ylabel(pair[1])

    init = False

n = ax.shape[0]
m = ax.shape[1]

for a in range(n):
    for b in range(m):
        ax[a,b].legend()
    
plt.show()

#### Data preparation

As NNs work with numbers and not strings, we will have to translate these properties' values into numbers. So let's create mappings for these properties. Here we will do it by creating lists for the respective properties, with the list index of the respective element serving as it's numerical mapping.

In [ ]:
mapping_species = list()

for species in penguins_dropanyna["species"].unique():
    mapping_species.append(f"{species}")

#mapping_species_sex = list()

#for species in penguins_dropanyna["species"].unique():
#    for sex in penguins_dropanyna["sex"].unique():
#        mapping_species_sex.append(f"{species} - {sex}")

In [ ]:
print(mapping_species)

In [ ]:
mapping_sex = list()

for sex in penguins_dropanyna["sex"].unique():
    mapping_sex.append(sex)

print(mapping_sex)

In [ ]:
mapping_islands = list()

for island in penguins_dropanyna["island"].unique():
    mapping_islands.append(island)

print(mapping_islands)

Now we'll prepare our data for use in our NN. Because we will make our NN discern between species through the 4 numerically measured bodily features, we first create a combined array containing the species' index and the bodily features.

In [ ]:
penguins_array = np.zeros((len(penguins_dropanyna), 5))

In [ ]:
for n, entry in penguins_dropanyna.iterrows():
    penguins_array[n, 0] = entry["bill_length_mm"]
    penguins_array[n, 1] = entry["bill_depth_mm"]
    penguins_array[n, 2] = entry["flipper_length_mm"]
    penguins_array[n, 3] = entry["body_mass_g"]

    search = f"{entry["species"]}"
    get_index = mapping_species.index(search)
    penguins_array[n, 4] = float(get_index)

    del search, get_index

In [ ]:
# mapping of the bodily features
property_mapping = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

In [ ]:
for n in range(penguins_array.shape[1]):
    print(penguins_array[:,n].max())

As you can see, the values for the measured features span different orders of magnitude. Because NNs usually train better on normalized data, we're going to scale down our features and keep the respective scaling values, because if we feed new data to our NN at some later point, the scaling factor must stay the same! Differently scaled data likely will lead to erroneous results!

Note also that we don't scale down our species indexes, because they are gonna be our target outputs.

In [ ]:
def lazy_scaling(array, t_indexes):
    maxes = dict()
    for index in t_indexes:
        tmp_max = array[:,index].max()
        lazy_max = 10*(math.ceil(tmp_max/10))
        array[:, index] /= lazy_max
        
        key = f"Index {index}"
        maxes[key] = lazy_max

        print(array[:,index].max())

    return array, maxes

In [ ]:
penguins_array, m_dict = lazy_scaling(penguins_array, t_indexes=[0,1,2,3])

In [ ]:
for n in range(penguins_array.shape[1]):
    print(penguins_array[:,n].max())

Now we're using scikit's `train_test_split` function to separate our dataset into training, validation and test sets.

Note that the default mode of the `train_test_split` function shuffles the data. Because we don't pass a distinct seed to the function the shuffle will (very likely) be different every time we execute the function.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
data, target = np.split(penguins_array, axis=1, indices_or_sections=[4])

In [ ]:
def prepare_data(data_array, target_array, test_perc=0.1, valid_perc=0.2):

    assert(data_array.shape[0] == target_array.shape[0]) ,\
    "Axis 0 shapes of data_array and target_array do not match!"

    data_temp, test_data, target_temp, test_target = train_test_split(data_array, target_array, test_size=test_perc)
    train_data, valid_data, train_target, valid_target = train_test_split(data_temp, target_temp, test_size=valid_perc)

    return(train_data, train_target, valid_data, valid_target, test_data, test_target)

In [ ]:
train_data, train_target, valid_data, valid_target, test_data, test_target = prepare_data(data, target)

Now let's have a quick look at the training and validation sets to make sure we've got a good spread of the data.

In [ ]:
fig, ax = plt.subplots(2,3)
fig.set_size_inches(15,10)

for n, pair in enumerate(key_pairs):
    x_index = property_mapping.index(pair[0])
    y_index = property_mapping.index(pair[1])
    
    ax[(n//3), (n%3)].scatter(x = train_data[:, x_index]*m_dict[f"Index {x_index}"],
                              y = train_data[:, y_index]*m_dict[f"Index {y_index}"],
                              label = "training", alpha=0.3
                             )
    ax[(n//3), (n%3)].scatter(x = valid_data[:, x_index]*m_dict[f"Index {x_index}"],
                              y = valid_data[:, y_index]*m_dict[f"Index {y_index}"],
                              label = "valid", alpha=0.3
                             )
    
    ax[(n//3), (n%3)].set_xlabel(pair[0])
    ax[(n//3), (n%3)].set_ylabel(pair[1])
    ax[(n//3), (n%3)].legend()

plt.legend()
plt.show()

### Creating the NNs

Now we're at a point where further preparation of data is specific to the used framework. So that part is moved to the respective section before the creation of the NN.

Another tool, supported by both PyTorch and Tensorflow, is Tensorboard. Tensorboard enables easy logging of NN parameters during training and allows for quick access to plots and more on a dashboard. This provides the user with quick and easy comparison of different training runs to e.g. spot with which parameters the model trains better/faster. We will keep to a very basic introduction of Tensorboard's functionality. Once you finished training with PyTorch and Tensorflow, you can open the Notebook `Tensorboard_view.ipynb` to have a look at the Tensorboard.

We also provide you with example classes for a simple (graphical) evaluation of your models.

The `Performance_Evaluator` class for plotting of false positive and false negative class predictions:

In [ ]:
class Performance_Evaluator:
    def __init__(self):
        self.error_dict = dict()

    def determine_errors(self, predictions, targets, setname: str, prop_mapping):
        self.mapping = prop_mapping
        num_c = len(self.mapping)
        self.error_dict[setname] = np.zeros((num_c, 2))
        for n, pred in enumerate(predictions):
            if pred != targets[n]:
                self.error_dict[setname][pred,0] += 1.
                self.error_dict[setname][targets[n],1] += 1.

        for c in range(num_c):
            ids, counts = np.unique(targets, return_counts=True)
            self.error_dict[setname][c,0] *= 100. / (np.sum(counts) - counts[c])
            self.error_dict[setname][c,1] *= -100. / counts[c]

    def plot_errors(self):
        fig, ax = plt.subplots(1, len(self.mapping))
        fig.set_size_inches(len(self.mapping)*5,5)

        print(ax, ax.shape)
        for n, species in enumerate(self.mapping):
            false_pos = np.zeros((len(self.error_dict.keys())))
            false_neg = np.zeros((len(self.error_dict.keys())))

            for k, label in enumerate(self.error_dict.keys()):
                false_pos[k] = self.error_dict[label][n,0]
                false_neg[k] = self.error_dict[label][n,1]

            ax[n].bar(np.arange(0, len(false_pos)), false_pos, label="False Positive")
            ax[n].bar(np.arange(0, len(false_neg)), false_neg, label="False Negative")
            ax[n].set_title(species)
            ax[n].set_xticks(np.arange(len(self.error_dict.keys())), self.error_dict.keys())
            ax[n].legend()
            ax[n].grid(visible=True, axis="y")

        ax[0].set_ylabel("Percentile Deviation")
        plt.show()

The `Model_Evaluator` class, serving as parent class for plotting the properties of predictions in comparison to the plots of the entire dataset.

In [ ]:
class Model_Evaluator:
    def __init__(self, model, dataset, pred_property, mapping):
        self.model = model
        self.ds = dataset
        self.prpr = pred_property
        self.mapping = mapping

    def plot_eval(self, prop_mapping):
        mask = np.ones(self.data[:self.num_range].shape, dtype=bool)
        mask[self.mask_ids, :] = False
        tmp_set = self.data[:self.num_range]
        matched_set = np.reshape(tmp_set[mask], shape=(-1,4))
        
        mask = np.invert(mask)
        mismatched_set = np.reshape(tmp_set[mask], shape=(-1,4))

        groups = self.ds.groupby(self.prpr)

        fig, ax = plt.subplots(2,3)
        fig.set_size_inches(15,10)
        init = True
        
        for name, group in groups:    
            for n, pair in enumerate(key_pairs):
                ax[(n//3), (n%3)].scatter(x = group[pair[0]], y = group[pair[1]], label = name, alpha = 0.3)
                
                if init == True:
                    ax[(n//3), (n%3)].set_xlabel(pair[0])
                    ax[(n//3), (n%3)].set_ylabel(pair[1])
        
            init = False
        
        for n, pair in enumerate(key_pairs):
            x_index = prop_mapping.index(pair[0])
            y_index = prop_mapping.index(pair[1])
            
            ax[(n//3), (n%3)].scatter(x = matched_set[:, x_index]*m_dict[f"Index {x_index}"],
                                      y = matched_set[:, y_index]*m_dict[f"Index {y_index}"],
                                      label = "matched", alpha=0.7, color="cyan")
            if np.size(mismatched_set, axis=0) > 0:
                ax[(n//3), (n%3)].scatter(x = mismatched_set[:, x_index]*m_dict[f"Index {x_index}"],
                                          y = mismatched_set[:, y_index]*m_dict[f"Index {y_index}"],
                                          label = "mismatched", alpha=0.7, color="red")

            ax[(n//3), (n%3)].legend()

        plt.show()

#### PyTorch

**Exercise Setup:**
- Begin with turning the data into PyTorch tensors.
- Recommended but optional: create and use TensorDatasets
- Use the cross entropy function as loss function
- When calling the loss function, pass parameter reduction="sum"
- Determine how many neurons the input and output layers must contain
- Create the sequential NN with the first hidden layer containing 16 neurons and a second hidden layer containing 8 neurons
- Use the ReLU function as activation function for the hidden layers
- Use the stochastic gradient descent optimizer (torch.optim.SGD) and pass momentum=momentum as final keyword argument

In [ ]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

So now, build your model and create cells as needed:

In [ ]:
# your code goes here

**Exercise Training:**

Define the training sequence, ideally containing a validation step for each epoch:
-Pass the writer object to the training function. At the end of each epoch, log the current loss using following command: <br>
`pt_writer.add_scalar("Loss/train", loss[0], epoch)`

The first attribute string "Loss/train" is the label assigned to the log, while 'epoch' tracks the current epoch.

You can also track additional scalars if you determine them. E.g. if you determine a validation loss, you can use the function from the previous command to track that as well using a different label.

Next perform the training with following parameters:
- learning rate = 5e-4
- batch size = 40
- epochs = 120
- momentum = 0.9

After training use test and/or validation sets to see how your model performs <br>
*optional:* try out different learning parameters. <br>
**Change the run_identifier between runs to avoid overwriting previous data!**

We create a writer object for logging the data. We determine the dictionary in which the data is to be stored. As a default it writes to directory runs, creating a subdirectory with the datetime of the object creation. We however want to write to directory logs and define our subdirectory ourself.

In [ ]:
# pass a unique identifier to the model training run. This will be the
# subdirectory name
run_identifier = "PyTorch_1"
pt_writer = SummaryWriter(log_dir=f"./output/logs/{run_identifier}")

Now it is time to train your model:

In [ ]:
# you code for model training goes here

Now we add some hyper_parameters to our writer object to have them available for inspection in Tensorboard:

In [ ]:
# Modify input as indicated and/or needed
hyper_parameters = dict()
hyper_parameters["learning rate"] = # your learning rate variable
hyper_parameters["batch size"] = # your batch size variable
hyper_parameters["momentum"] = # your momentum variable

metrics = dict()

pt_writer.add_hparams(hparam_dict=hyper_parameters, metric_dict=metrics)

pt_writer.close()

del hyper_parameters, metrics

Now that you finished your model creation and training, we create a child class to our `Model_Evaluator` parent class, adding a function that creates predictions and compares them to the actual data.

In [ ]:
class ME_PyTorch(Model_Evaluator):
    def __init__(self, model, dataset, pred_property, mapping):
        super().__init__(model, dataset, pred_property, mapping)

    def predict(self, data, targets, num_range=None):        
        self.mask_ids = list()
        self.data = data
        self.targets = targets
        
        if num_range != None:
            self.num_range = num_range
        else:
            self.num_range = self.targets.shape[0]

        model.eval()
        self.pred = model(self.data[:num_range])

        for n in range(self.num_range):
            pred = self.pred[n]
            if torch.argmax(pred) != self.targets[n]:    
                colour = "\033[95m" 
                self.mask_ids.append(n)
                          
            else:
                colour = "\033[92m"
            
            print(f"{colour} {n} Prediction is: {torch.argmax(pred)} - {self.mapping[torch.argmax(pred)]} ; "
            + f"Actual data says: {self.targets[n]} - {self.mapping[self.targets[n]]} | "
            + f"Probabilities {torch.nn.functional.softmax(pred, dim=0).data}\033[00m")

        print(f"{len(self.mask_ids)} samples were mismatched")

Object creation:

In [ ]:
pt_modev = ME_PyTorch(model, penguins_dropanyna, "species", mapping_species)

pt_perf = Performance_Evaluator()

##### Training set

*Optional*

##### Validation set

In [ ]:
pt_modev.predict(valid_x, valid_y)

In [ ]:
pt_modev.plot_eval(property_mapping)

In [ ]:
pt_perf.determine_errors(torch.argmax(pt_modev.pred.data, axis=1), valid_y[:pt_modev.num_range], "Validation", mapping_species)

##### Test set

In [ ]:
pt_modev.predict(test_x, test_y)

In [ ]:
pt_modev.plot_eval(property_mapping)

In [ ]:
pt_perf.determine_errors(torch.argmax(pt_modev.pred.data, axis=1), test_y[:pt_modev.num_range], "Test", mapping_species)

In [ ]:
pt_perf.plot_errors()

#### Tensorflow

**Exercise Setup:**
- Begin with turning the data into PyTorch tensors.
- Use the cross entropy function as loss function (SparseCategoricalCrossentropy)
- When calling the loss function, pass parameter reduction="sum"
- Make sure input and output layers contain the correct number of nodes
- Create the sequential NN with the first hidden layer containing 16 neurons and a second hidden layer containing 8 neurons
- Use the ReLU function as activation function for the hidden layers
- Use the stochastic gradient descent optimizer (tf.keras.optimizers.SGD) and pass momentum=momentum as final keyword argument

In [ ]:
# TensorFlow and tf.keras
import tensorflow as tf

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
CUDA_VISIBLE_DEVICES=""

So now, build your model and create cells as needed:

In [ ]:
# your code goes here

**Exercise Training:**

Define the training sequence, ideally containing a validation step for each epoch:

Next perform the training with following parameters:
- learning rate = 5e-4
- batch size = 40
- epochs = 120
- momentum = 0.9

After training use test and/or validation sets to see how your model performs <br>
*optional:* try out different learning parameters. <br>
**Change the run_identifier between runs to avoid overwriting previous data!**

In [ ]:
tf_run_id = "Tensorflow_1"

tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=f"./output/logs/{tf_run_id}", histogram_freq=1)

To enable logging in Tensorflow, add the keyword argument `callbacks=[tensorboard_callback]` to the fit function.

In [ ]:
# your code for training goes here

In [ ]:
class ME_Tensorflow(Model_Evaluator):
    def __init__(self, model, dataset, pred_property, mapping):
        super().__init__(model, dataset, pred_property, mapping)

    def predict(self, data, targets, num_range=None):        
        self.mask_ids = list()
        self.data = data
        self.targets = targets
        
        if num_range != None:
            self.num_range = num_range
        else:
            self.num_range = self.targets.shape[0]

        self.pred = model(self.data[:self.num_range])
        for n in range(self.num_range):
            pred = self.pred[n]
            if np.argmax(pred) != self.targets[n]:    
                colour = "\033[95m" 
                self.mask_ids.append(n)
                          
            else:
                colour = "\033[92m"

            print(f"{colour} {n} Prediction is: {np.argmax(pred)} - {self.mapping[np.argmax(pred)]} ; "
            + f"Actual data says: {self.targets[n]} - {self.mapping[int(self.targets[n][0])]} | "
            + f"Probabilities {tf.nn.softmax(pred)}\033[00m")

        print(f"{len(self.mask_ids)} samples were mismatched")

In [ ]:
tf_modev = ME_Tensorflow(model, penguins_dropanyna, "species", mapping_species)

tf_perf = Performance_Evaluator()

##### Training set

*Optional*

##### Validation set

In [ ]:
tf_modev.predict(valid_data, valid_target)

In [ ]:
tf_modev.plot_eval(property_mapping)

In [ ]:
tf_perf.determine_errors(np.argmax(tf_modev.pred, axis=1), np.array(valid_target[:tf_modev.num_range], dtype=int), "Validation", mapping_species)

##### Test set

In [ ]:
tf_modev.predict(test_data, test_target)

In [ ]:
tf_modev.plot_eval(property_mapping)

In [ ]:
tf_perf.determine_errors(np.argmax(tf_modev.pred, axis=1), np.array(test_target[:tf_modev.num_range], dtype=int), "Test", mapping_species)

In [ ]:
tf_perf.plot_errors()

### Resources

Introduction to Artificial Neural Networks:

- https://youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi&si=K6NmU277knsiknd7

- https://www.mzes.uni-mannheim.de/socialsciencedatalab/article/ann/

- https://www.geeksforgeeks.org/artificial-neural-networks-and-its-applications/

  

Autoencoders:

- https://towardsdatascience.com/introduction-to-autoencoders-7a47cf4ef14b

- https://www.tensorflow.org/tutorials/generative/autoencoder

- https://www.datacamp.com/tutorial/introduction-to-autoencoders

  

Convolutional Neural Networks:

- https://saturncloud.io/blog/a-comprehensive-guide-to-convolutional-neural-networks-the-eli5-way/

- https://www.youtube.com/watch?v=KuXjwB4LzSA&t=363s

- https://www.youtube.com/watch?v=py5byOOHZM8


Materials & Tutorials:

- https://www.tensorflow.org/tutorials/

- https://ki-kurs.org/ Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz

- https://huggingface.co/ Platform for tools for the creation of applications with machine learning.


Mixed - to be sorted:

- https://stackoverflow.com/a/27134600

- https://alexlenail.me/NN-SVG/index.html (creating NN-structure graphics)

- https://pytorch.org/tutorials/beginner/data_loading_tutorial.html -> example walkthrough creating a custom `FacialLandmarkDataset` class as a subclass of `Dataset`.

- https://pytorch.org/docs/stable/_modules/torch/utils/data/dataset.html#TensorDataset

- https://www.fast.ai/2017/11/13/validation-sets/ -> on validation

- https://www.quora.com/Does-the-order-of-training-data-matter-when-training-neural-networks -> on data shuffling

You can also host your own Jupyter Server on the cluster nodes.

You can do this either by submitting a corresponding batch script to the scheduling system, or by using an interactive session.

The following step-by-step guide illustrates how to start your own JupyterLab on the bwUniluster

1. Login to the bwUniCluster via a command-line interface (CLI)

2. Allocate resources for the interactive jobs with salloc on a login node: 

    `$ salloc -p single -n 1 --ntasks-per-node=40 -t 24:00:00 --mem=20gb`

    You can adjust the resources to your needs.

3. After your interactive session has started you can load the required modules and run a Jupyterlab session without a browser: 

    `$ module load jupyter/base`

    `$ jupyter lab --no-browser --port=8888 --ip 0.0.0.0`
    
4. Set up a SHH tunnel on your local machine (you need to change COMPUTENODE and USER accordingly):

    `$ ssh -L 8888:<COMPUTENODE>:8888 <USER>@uc2.scc.kit.edu`
    
    Connect to the Jupyter session from your web browser: http://127.0.0.1:8888 (see console output for more information)

**Note:** if you work on the bwForCluster BinAC check out https://wiki.bwhpc.de/e/BinAC/Software/Jupyterlab

It is strongly recommended to use virtual environments when working with Python.

Having separate environments for different research projects is very convenient, as it allows you to have different versions of the same package, better handle dependencies, and
crashing an environment is preferable to messing up your base Python environment. 

There are two options for how you can set up your environments, either via pythons built-in virtual environment 'venv' module (see section 8 at https://wiki.bwhpc.de/e/BwUniCluster2.0/Jupyter) or via the 'devel/miniconda' module. 

You can find a short introduction on how to use conda for the bwForCluster Helix here: https://wiki.bwhpc.de/e/Helix/Software/Conda, it works similarly on the bwUniCluster, and usually comprises the following steps:

`$ conda create -n <env_name>`

`$ conda activate <env_name>`

`$ conda install -c <channel> <package>`

To use the environment in your Jupyter Notebook, you have to register it as a Jupyter Kernel via the ipykernel package:

`$ python -m ipykernel install --user --name <env_name> --display-name "<Displayed Kernel Name>"`

Inside JupyterLab, you can switch kernels using the toolbar (try it out in this Notebook!)

Please complete the **bwHPC symposium evaluation**.

We would love your feedback on this workshop, as it will help us improve it for future events!

# References

The content of this workshop is in parts based on and inspired by the following sources:

* Python Course of the AG Peter (Prof. Dr. Christine Peter, Kevin Savade, Dr. Oleksandra Kukharenko, Dr. Andrej Berg)
* Software Carpentry workshops (https://software-carpentry.org/lessons/)
* Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz (https://ki-kurs.org/)
* Real Python (https://realpython.com/)
* Intro to Autoencoders (https://www.tensorflow.org/tutorials/generative/autoencoder)
* Image classification of MNIST using TensorFlow (https://www.kaggle.com/code/viratkothari/image-classification-of-mnist-using-tensorflow)
* The bwHPC wiki (https://wiki.bwhpc.de/)